In [ ]:

!pip -q install ipywidgets pandas
from google.colab import output
output.enable_custom_widget_manager()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.2 MB/s eta 0:00:00


In [ ]:

import os
from typing import Any, Dict, Optional
import ipywidgets as widgets
from IPython.display import display, HTML


class OrchestrationClient:
    """
    UI-to-orchestration adapter.
    Replace `call_orchestrator` with your FastAPI / LangChain / LlamaIndex / agentic RAG call.
    Expected response:
    {
        "summary": [
            "<b>Query received:</b> ...",
            "<b>Dataset identified:</b> ...",
            "<b>Business owner:</b> ..."
        ]
    }
    """

    def __init__(self, endpoint_url: Optional[str] = None):
        self.endpoint_url = endpoint_url or os.getenv("ORCHESTRATION_URL", "")

    def call_orchestrator(self, user_query: str, uploaded_file: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        # Replace this mock response with a real orchestration-layer call.
        # Example integration choices:
        # 1. requests.post(self.endpoint_url, json=payload)
        # 2. direct import from orchestration service
        # 3. LangChain/LlamaIndex agent call
        summary = [
            f"<b>Query received:</b> {user_query}",
            "<b>Dataset identified:</b> home_equity.",
            "<b>Business owner:</b> Home Equity Analytics.",
            "<b>Technical owner:</b> Mortgage Data Platform.",
            "<b>LOBs using this dataset:</b> Home Lending.",
            "<b>Pipeline status:</b> Healthy.",
            "<b>Freshness:</b> 2 hours.",
            "<b>Completeness:</b> 98.7%.",
            "<b>Upstream sources:</b> sas.he_base, servicing.payments.",
            "<b>Downstream targets:</b> lake.he_curated, risk.he_dashboard."
        ]
        if uploaded_file:
            summary.append(f"<b>Uploaded file processed:</b> {uploaded_file.get('name', 'uploaded_file')}.")
        return {"summary": summary}


class GenAIDataDiscoveryUI:
    def __init__(self, orchestrator_client: OrchestrationClient):
        self.orchestrator_client = orchestrator_client
        self.feedback_log = []
        self._build_ui()

    def _inject_styles(self) -> None:
        display(HTML("""
        <style>
        .genai-shell{
          border:1px solid #e5e7eb;border-radius:18px;padding:18px;
          background:linear-gradient(180deg,#ffffff 0%,#f8fbff 100%);
          box-shadow:0 8px 24px rgba(15,23,42,.08);font-family:Arial,sans-serif;margin-bottom:40px;
        }
        .genai-title{font-size:24px;font-weight:700;color:#1f2937;margin-bottom:6px;}
        .genai-subtitle{font-size:13px;color:#6b7280;}
        .output{
          border:1px solid #dbe4f0;border-radius:16px;padding:20px;background:#fafcff;
        }
        .query{
          background:#eef6ff;padding:12px;border-radius:10px;margin-bottom:16px;
        }
        .summary-title{
          font-size:14px;font-weight:700;color:#1d4ed8;margin-bottom:12px;
        }
        .summary-item{
          font-size:13px;color:#334155;line-height:1.8;margin-bottom:12px;
        }
        .placeholder{color:#64748b;font-size:13px;padding:18px 8px;}
        .small-note{font-size:12px;color:#64748b;margin-top:8px;}
        .white-btn button{
          background:white !important;
          border:1px solid #d1d5db !important;
          color:#111827 !important;
          box-shadow:none !important;
        }
        </style>
        """))

    def _parse_uploaded_widget_file(self, file_item):
        if isinstance(file_item, dict):
            return {"name": file_item.get("name", "uploaded_file"), "content": file_item.get("content", b"")}
        return {"name": getattr(file_item, "name", "uploaded_file"), "content": getattr(file_item, "content", b"")}

    def _get_uploaded_payload(self):
        if not self.upload.value:
            return None
        if isinstance(self.upload.value, dict):
            first_item = next(iter(self.upload.value.values()))
        else:
            first_item = list(self.upload.value)[0]
        return self._parse_uploaded_widget_file(first_item)

    def _build_output_html(self, response: Dict[str, Any], user_query: str) -> str:
        summary = response.get("summary", [])
        summary_items = summary if isinstance(summary, list) else [summary]
        summary_html = "".join([f"<div class='summary-item'>{item}</div>" for item in summary_items])
        return f"""
        <div class='output'>
          <div class='query'><b>User Query:</b> {user_query}</div>
          <div class='summary-title'>Summary</div>
          {summary_html}
        </div>
        """

    def _handle_submit(self, _):
        user_query = self.query_box.value.strip() or "Analyze uploaded file"
        uploaded = self._get_uploaded_payload()
        response = self.orchestrator_client.call_orchestrator(
            user_query=user_query,
            uploaded_file=uploaded
        )
        self.output_html.value = self._build_output_html(response, user_query)
        self.status_bar.value = "<span style='color:#2e7d32;font-weight:600;'>Response rendered successfully.</span>"

    def _handle_clear(self, _):
        self.query_box.value = ""
        self.upload.value = ()
        self.output_html.value = "<div class='placeholder'>Your response will appear here.</div>"
        self.status_bar.value = "<span style='color:#666;'>Cleared.</span>"

    def _handle_thumbsup(self, _):
        self.feedback_log.append({"feedback":"up","query":self.query_box.value})
        self.feedback_status.value = "<span style='color:#2e7d32;font-weight:600;'>Thanks for the positive feedback.</span>"

    def _handle_thumbsdown(self, _):
        self.feedback_log.append({"feedback":"down","query":self.query_box.value})
        self.feedback_status.value = "<span style='color:#c62828;font-weight:600;'>Feedback captured. You can refine prompts or orchestration logic.</span>"

    def _build_ui(self):
        self._inject_styles()

        self.header = widgets.HTML("""
        <div class='genai-shell'>
          <div class='genai-title'>GenAI Data Discovery Assistant</div>
          <div class='genai-subtitle'>Ask about metadata, lineage, health, ownership, or upload a file for quick profiling.</div>
        </div>
        """)

        self.query_box = widgets.Textarea(
            value="Show lineage and health of home_equity dataset",
            placeholder="Type your natural language query here...",
            description="Enter your question here",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="100%", height="100px")
        )

        self.upload = widgets.FileUpload(
            accept=".pdf,.csv,.txt,.md,.log,.json",
            multiple=False,
            description="Upload file",
            layout=widgets.Layout(width="auto")
        )

        self.submit_btn = widgets.Button(description="Submit", button_style="success", icon="check", layout=widgets.Layout(width="auto"))
        self.clear_btn = widgets.Button(description="Clear", icon="trash", layout=widgets.Layout(width="auto"))
        self.clear_btn.add_class("white-btn")

        self.thumbsup_btn = widgets.Button(description="Helpful", icon="thumbs-up")
        self.thumbsdown_btn = widgets.Button(description="Not Helpful", icon="thumbs-down")
        self.thumbsup_btn.add_class("white-btn")
        self.thumbsdown_btn.add_class("white-btn")

        self.status_bar = widgets.HTML("<span style='color:#666;'>Ready.</span>")
        self.feedback_status = widgets.HTML("<span style='color:#666;'>Feedback not submitted yet.</span>")
        self.output_html = widgets.HTML("<div class='placeholder'>Your response will appear here.</div>")

        self.submit_btn.on_click(self._handle_submit)
        self.clear_btn.on_click(self._handle_clear)
        self.thumbsup_btn.on_click(self._handle_thumbsup)
        self.thumbsdown_btn.on_click(self._handle_thumbsdown)

        self.input_panel = widgets.Box(
            [self.query_box],
            layout=widgets.Layout(border="1px solid #e5e7eb", padding="24px", border_radius="16px", margin="0 0 0 0", width="100%")
        )

        upload_with_note = widgets.VBox(
            [
                self.upload,
                widgets.HTML("<div class='small-note'>pdf, csv, txt, md, log, json</div>")
            ],
            layout=widgets.Layout(gap="0px", align_items="flex-start")
        )

        controls_row = widgets.HBox(
            [upload_with_note, self.clear_btn, self.submit_btn],
            layout=widgets.Layout(
                gap="10px",
                align_items="flex-start",
                justify_content="flex-end",
                width="100%"
            )
        )

        self.controls_panel = widgets.Box(
            [controls_row],
            layout=widgets.Layout(border="1px solid #e5e7eb", padding="24px", border_radius="16px", margin="0 0 40px 0", width="100%")
        )

        self.output_panel = widgets.Box(
            [self.output_html],
            layout=widgets.Layout(border="1px solid #e5e7eb", padding="24px", border_radius="16px", margin="0 0 40px 0")
        )

        self.feedback_row = widgets.HBox(
            [self.thumbsup_btn, self.thumbsdown_btn],
            layout=widgets.Layout(gap="10px", margin="0 0 24px 0")
        )

        self.layout = widgets.VBox([
            self.header,
            self.input_panel,
            self.controls_panel,
            self.status_bar,
            self.output_panel,
            self.feedback_row,
            self.feedback_status
        ], layout=widgets.Layout(width="100%"))

    def render(self):
        display(self.layout)


def create_ui(orchestration_url: Optional[str] = None):
    client = OrchestrationClient(endpoint_url=orchestration_url)
    ui = GenAIDataDiscoveryUI(client)
    ui.render()
    return ui


In [ ]:

# Render the UI
ui = create_ui()
